# SI 618 Project

## Data Manipulation

In this step, the main things we want to do are: 
1. data filtering: The data we need is 
    - weather data from 2023-05-01 to 2024-05-01
    - usage_frequency data from 2023-05-01 to 2024-05-31
    - daily rent data from 2024-05-01 to 2024-05-31
    - station_list data
2. data format conversion
3. data merging
4. basic data cleaning

### Loading Packages and Dataset

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
datetag1 = '2023-05-01 00:00:00'
datetag2 = '2024-05-01 00:00:00'
datetag3= '2024-05-31 23:59:59'


In [29]:
usage_frequency=pd.read_csv('bikeshare/usage_frequency.csv')
station_list=pd.read_csv('bikeshare/station_list.csv')
weather=pd.read_csv('bikeshare/weather.csv')

Our original dataset contains a table showing the rental records for each stations. This table has too much data, so we perform an initial data filter at the data import stage. 

We chose data from 2024-05-01 to 2024-05-31 to ensure that our dataset would be current while still covering the fluctuating changes in rentals over a full week (we initially assume that there might be some differences in the bikeshare data during the week and on weekends).

In [30]:
# this step will output a filtered csv file
# we will use the output file in the following steps

# date_cols = ['started_at']
# data = pd.read_csv('bikeshare/daily_rent_detail.csv', parse_dates=['started_at'])

# filtered_data = data[(data['started_at'] >= datetag2) & (data['started_at'] <= datetag3)]
# filtered_data.to_csv('bikeshare/daily_rent_detail_240531.csv', index=False)

In [31]:
rent_data = pd.read_csv('bikeshare/daily_rent_detail_240531.csv')

### Data Display

In [32]:
usage_frequency.head()

,date,station_name,pickup_counts,dropoff_counts
0,2020-05-01,10th & E St NW,11,7.0
1,2020-05-01,10th & Florida Ave NW,8,8.0
2,2020-05-01,10th & G St NW,3,2.0
3,2020-05-01,10th & K St NW,12,15.0
4,2020-05-01,10th & Monroe St NE,5,6.0


In [33]:
usage_frequency.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 873318 entries, 0 to 873317
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   date            873318 non-null  object 
 1   station_name    873318 non-null  object 
 2   pickup_counts   873318 non-null  int64  
 3   dropoff_counts  873318 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 26.7+ MB


In [34]:
rent_data.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,028CB30D63626320,classic_bike,2024-05-23 16:07:47,2024-05-23 16:20:22,Grant Circle,31421.0,10th & Quincy St NE / Turkey Thicket Rec,31541.0,38.942025,-77.018221,38.937849,-76.993509,casual
1,33D6F7F8951D5D67,electric_bike,2024-05-24 09:19:05,2024-05-24 10:02:29,W Columbia St & N Washington St,32609.0,S Glebe Rd & Potomac Ave,31010.0,38.885602,-77.166884,38.842600,-77.050200,casual
2,BDDEFEBB8770EFF2,classic_bike,2024-05-09 08:54:14,2024-05-09 08:59:16,1st & L St NW,31677.0,New Jersey Ave & F St NW,31655.0,38.903819,-77.011987,38.897108,-77.011616,member
3,31C46CFE02542EF5,classic_bike,2024-05-16 13:48:38,2024-05-16 14:04:43,North Capitol & R St NE,31527.0,10th & Quincy St NE / Turkey Thicket Rec,31541.0,38.912560,-77.008775,38.937849,-76.993509,member
4,3DB767AB9DF69BD7,electric_bike,2024-05-22 08:27:00,2024-05-22 08:38:24,Eastern Market / 7th & North Carolina Ave SE,31610.0,New Jersey Ave & F St NW,31655.0,38.887016,-76.996802,38.897108,-77.011616,member


In [35]:
rent_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515535 entries, 0 to 515534
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             515535 non-null  object 
 1   rideable_type       515535 non-null  object 
 2   started_at          515535 non-null  object 
 3   ended_at            515535 non-null  object 
 4   start_station_name  436789 non-null  object 
 5   start_station_id    436789 non-null  float64
 6   end_station_name    433527 non-null  object 
 7   end_station_id      433407 non-null  float64
 8   start_lat           515535 non-null  float64
 9   start_lng           515535 non-null  float64
 10  end_lat             515113 non-null  float64
 11  end_lng             515113 non-null  float64
 12  member_casual       515535 non-null  object 
dtypes: float64(6), object(7)
memory usage: 51.1+ MB


In [36]:
weather.head()

,name,datetime,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,...,solarenergy,uvindex,severerisk,sunrise,sunset,moonphase,conditions,description,icon,stations
0,"Washington,DC,USA",2020-05-01,18.8,11.6,14.9,18.8,11.6,14.9,8.9,69.6,...,9.5,6,NaN,2020-05-01T06:09:41,2020-05-01T20:01:16,0.30,"Rain, Partially cloudy",Partly cloudy throughout the day with rain cle...,rain,"KDCA,72405013743,72403793728,F0198,KADW,KDAA,7..."
1,"Washington,DC,USA",2020-05-02,22.1,11.1,16.3,22.1,11.1,16.3,6.4,54.0,...,14.0,9,NaN,2020-05-02T06:08:30,2020-05-02T20:02:13,0.33,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"KDCA,72405013743,72403793728,F0198,KGAI,KADW,K..."
2,"Washington,DC,USA",2020-05-03,24.9,15.6,18.6,24.9,15.6,18.6,13.4,72.5,...,5.7,4,NaN,2020-05-03T06:07:21,2020-05-03T20:03:11,0.37,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,"KDCA,72405013743,72403793728,F0198,KADW,KDAA,7..."
3,"Washington,DC,USA",2020-05-04,23.8,14.3,19.2,23.8,14.3,19.2,7.8,53.8,...,13.3,8,NaN,2020-05-04T06:06:12,2020-05-04T20:04:08,0.40,"Rain, Partially cloudy",Clearing in the afternoon with early morning r...,rain,"KDCA,72405013743,72403793728,F0198,KADW,KDAA,7..."
4,"Washington,DC,USA",2020-05-05,14.3,9.3,12.3,14.3,8.0,12.1,3.3,55.6,...,5.0,2,NaN,2020-05-05T06:05:06,2020-05-05T20:05:05,0.44,"Rain, Partially cloudy",Partly cloudy throughout the day with late aft...,rain,"KIAD,KDCA,72405013743,72403793728,72403093738,..."


In [37]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1584 entries, 0 to 1583
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              1584 non-null   object 
 1   datetime          1584 non-null   object 
 2   tempmax           1584 non-null   float64
 3   tempmin           1584 non-null   float64
 4   temp              1584 non-null   float64
 5   feelslikemax      1584 non-null   float64
 6   feelslikemin      1584 non-null   float64
 7   feelslike         1584 non-null   float64
 8   dew               1584 non-null   float64
 9   humidity          1584 non-null   float64
 10  precip            1584 non-null   float64
 11  precipprob        1584 non-null   int64  
 12  precipcover       1584 non-null   float64
 13  preciptype        757 non-null    object 
 14  snow              1584 non-null   float64
 15  snowdepth         1584 non-null   float64
 16  windgust          1584 non-null   float64


In [38]:
station_list.head()

,station_id,station_name
0,30200,9th St & Pennsylvania Ave NW
1,30201,9th & G St NW
2,31000,Eads St & 15th St S
3,31001,18th St & S Eads St
4,31002,Crystal Dr & 20th St S


In [39]:
station_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   station_id    916 non-null    int64 
 1   station_name  916 non-null    object
dtypes: int64(1), object(1)
memory usage: 14.4+ KB


### Preliminary data-processing

In this step, we did some basic data processing on all tables. This includes removing data from other times according to the study period, modifying the data types of some columns, etc.
We hope that after this step, the tables will be able to be merged.

- **For usage_frequency df**：We primarily intercepted data from the study time period. Since we may need to time period, we output data for a month (from 2024-05-01 to 2024-05-31) and a year (from 2023-05-01 to 2024-05-01).
- **For weather df**：We primarily intercepted data from the study time period (from 2023-05-01 to 2024-05-01). And we also remove some columns that we may not need. We have also changed the names of some of the columns for better merge.
- **For rent_data df**：We transfer the station code from float data to int data. During the checking process, we found that some pickup times contain milliseconds information, we would like to remove this useless information. We also extracted the day information from this time column and stored it in a new ‘started_day’ column for subsequent data merging.

In [40]:

usage_frequency['date'] = pd.to_datetime(usage_frequency['date'])

usage_frequency_m=usage_frequency.loc[(usage_frequency['date']>=datetag2) & (usage_frequency['date']<=datetag3)]
usage_frequency_m=usage_frequency_m.rename(columns={'station_name':'start_station_name','date':'started_day'})


usage_frequency_y=usage_frequency.loc[(usage_frequency['date']>=datetag1) & (usage_frequency['date']<=datetag2)]
usage_frequency_y.loc[:,'city_name']='Washington,DC,USA'

/var/folders/dd/httq2b3x1vx0862q77843f880000gn/T/ipykernel_10876/1239207563.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  usage_frequency_y.loc[:,'city_name']='Washington,DC,USA'


In [41]:
weather['datetime'] = pd.to_datetime(weather['datetime'])

weather=weather.loc[(weather['datetime']>=datetag1) & (weather['datetime']<=datetag2)]
weather=weather.iloc[:,:-1]
weather.rename(columns={'name':'city_name','datetime':'date'},inplace=True)


In [42]:
rent_data['started_at'] = rent_data['started_at'].str.split('.').str[0]
rent_data['ended_at'] = rent_data['ended_at'].str.split('.').str[0]

rent_data['started_at'] = pd.to_datetime(rent_data['started_at'], format="%Y-%m-%d %H:%M:%S", errors='coerce')
rent_data['ended_at'] = pd.to_datetime(rent_data['ended_at'], format="%Y-%m-%d %H:%M:%S", errors='coerce')
rent_data['started_day'] = rent_data['started_at'].dt.floor('D')


rent_data['start_station_id'] = rent_data['start_station_id'].where(rent_data['start_station_id'].notnull(), None).astype('Int64')
rent_data['end_station_id'] = rent_data['end_station_id'].where(rent_data['end_station_id'].notnull(), None).astype('Int64')

### Merging Data

In our analysis, we roughly merge the table into two dataframes based on the correlation between the data.
- (1) We selected one year of weather and merged the pickup & dropoff records of each site for each day of the year period with the site's current weather data.
- (2) We selected a detailed record of rentals for all sites over a month period and combined it with the total rentals for each site for each day. We mainly observed where they pick up the bikes,  so we merged the station data with the starting station of the detail records.

In [43]:
weather_frequency=pd.merge(usage_frequency_y,weather,on=['city_name','date'],how='left')

In [44]:
weather_frequency.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225801 entries, 0 to 225800
Data columns (total 35 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              225801 non-null  datetime64[ns]
 1   station_name      225801 non-null  object        
 2   pickup_counts     225801 non-null  int64         
 3   dropoff_counts    225801 non-null  float64       
 4   city_name         225801 non-null  object        
 5   tempmax           225801 non-null  float64       
 6   tempmin           225801 non-null  float64       
 7   temp              225801 non-null  float64       
 8   feelslikemax      225801 non-null  float64       
 9   feelslikemin      225801 non-null  float64       
 10  feelslike         225801 non-null  float64       
 11  dew               225801 non-null  float64       
 12  humidity          225801 non-null  float64       
 13  precip            225801 non-null  float64       
 14  prec

In [45]:
detail_frequency=rent_data.merge(usage_frequency_m,on=['start_station_name','started_day'],how='left')

In [46]:
detail_frequency.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515535 entries, 0 to 515534
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             515535 non-null  object        
 1   rideable_type       515535 non-null  object        
 2   started_at          515535 non-null  datetime64[ns]
 3   ended_at            515535 non-null  datetime64[ns]
 4   start_station_name  436789 non-null  object        
 5   start_station_id    436789 non-null  Int64         
 6   end_station_name    433527 non-null  object        
 7   end_station_id      433407 non-null  Int64         
 8   start_lat           515535 non-null  float64       
 9   start_lng           515535 non-null  float64       
 10  end_lat             515113 non-null  float64       
 11  end_lng             515113 non-null  float64       
 12  member_casual       515535 non-null  object        
 13  started_day         515535 no

After merging operation, we get two main tables: 

- **weather_frequency** table: this table contains the daily pickup & dropoff of all the rental stations in one year, as well as the weather condition of the day.
- **detail_frequency** table: this table contains the details of rental records (including rental time, start location, end location, end time, bike type, etc.) of all rental stations for each day in one month, and the overall rental situation of each day of every stations.

### Secondary data processing and data cleansing


After merging the data upfront, we get a lot of null values in our data. In this step we will perform a simple data cleanup on the two tables we merged.

The main part of our work is to work with the null values in the merged table.
- For the **weather_frequency** table, we found that the null values were mainly about the precipitation type. Since it's not the main data we want and the nulls make up a large percentage, we can't just remove them. Therefore, when doing the processing, we chose to fill them with 0.
- For the **detail_frequency** table, the null values are mainly about the start and end locations The records that lack these data cannot be analyzed as we want, so we directly delete this part of the record.


In [47]:
weather_frequency.isnull().sum()[weather_frequency.isnull().any()]

preciptype    114768
dtype: int64

In [48]:
weather_frequency.fillna(0,inplace=True)

In [49]:
weather_frequency.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225801 entries, 0 to 225800
Data columns (total 35 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              225801 non-null  datetime64[ns]
 1   station_name      225801 non-null  object        
 2   pickup_counts     225801 non-null  int64         
 3   dropoff_counts    225801 non-null  float64       
 4   city_name         225801 non-null  object        
 5   tempmax           225801 non-null  float64       
 6   tempmin           225801 non-null  float64       
 7   temp              225801 non-null  float64       
 8   feelslikemax      225801 non-null  float64       
 9   feelslikemin      225801 non-null  float64       
 10  feelslike         225801 non-null  float64       
 11  dew               225801 non-null  float64       
 12  humidity          225801 non-null  float64       
 13  precip            225801 non-null  float64       
 14  prec

In [50]:
detail_frequency.isnull().sum()[detail_frequency.isnull().any()]

start_station_name    78746
start_station_id      78746
end_station_name      82008
end_station_id        82128
end_lat                 422
end_lng                 422
pickup_counts         78746
dropoff_counts        78746
dtype: int64

In [51]:
detail_frequency.dropna(inplace=True)

In [52]:
detail_frequency.info()

<class 'pandas.core.frame.DataFrame'>
Index: 400814 entries, 0 to 515526
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             400814 non-null  object        
 1   rideable_type       400814 non-null  object        
 2   started_at          400814 non-null  datetime64[ns]
 3   ended_at            400814 non-null  datetime64[ns]
 4   start_station_name  400814 non-null  object        
 5   start_station_id    400814 non-null  Int64         
 6   end_station_name    400814 non-null  object        
 7   end_station_id      400814 non-null  Int64         
 8   start_lat           400814 non-null  float64       
 9   start_lng           400814 non-null  float64       
 10  end_lat             400814 non-null  float64       
 11  end_lng             400814 non-null  float64       
 12  member_casual       400814 non-null  object        
 13  started_day         400814 non-nul